In [ ]:
!pip install Pillow==9.5.0

In [ ]:
import torch
from torchvision import models
torch.manual_seed(0)
from torch.autograd import Variable
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import transforms
import os

from PIL import Image
import urllib
import seaborn as sns

#1. Wczytanie modelu do klasyfikacji

Na początek ćwiczenia wczytamy model wytrenowany na zbiorze ImageNet służący do zadania klasyfikacji obrazu. W tym cely wykorzystamy sieć VGG-16 ponieważ dobrze współpracuje ona z biblioteką do wizualizacji decyzji sieci CNN, ale przy odrobinie wysiłku można by wykorzystać dowolny model z model-zoo
https://pytorch.org/vision/stable/models.html

Wszystkie wytrenowane modele normalizują obraz treningowy tak aby średnia wartość pikseli wynosiła 0, a odchylenie standardowe 1. Kod poniżej wykonuje odpowiednią normalizację.

In [ ]:
model = models.vgg16(pretrained=True)

print(model.features)
#wczytanie wytrenowanego modelu

#transformacja obrazu do rozmiaru oczekiwanego przez sieć
preprocess_img = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
])

#transformacja obrazu wraz z normalizacją
preprocess = transforms.Compose([
    preprocess_img,
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

Model będzie zajmować się klasyfikacją, tak jak w konkursie ImageNet.
Przyjrzyj sie klasom znajdującym się pod adresem https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt
Kod poniżej tworzy słownik, który zwraca nazwę klasy dla id zwracanego przez sieć.

In [ ]:
# pobranie pliku
!wget https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt

with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

In [ ]:
# POBRANIE PRZYKŁADOWYCH OBRAZÓW

# Pobierzmy przykładowe obrazy
#img_link = "http://farm4.staticflickr.com/3523/3976430294_7664ccce39_z.jpg" #niedźwiedź
img_link = "https://github.com/pytorch/hub/raw/master/images/dog.jpg" #pies

url, filename = (img_link, "image.jpg")
try: urllib.URLopener().retrieve(url, filename)
except: urllib.request.urlretrieve(url, filename)

# Zadanie 1. [1 pkt.]
Sieć zwraca nieznormalizowaną wartość dla każdej z klas. Zamień wartości zwracane przez sieć tak aby określały one **pewność (prawdopodobieństwo) sieci dla każdej z klas**. Szukaną funkcję znajdziesz w bibliotece torch.

W konkursie ImageNet standardowo patrzy się na 5 "najpewniejszych" klas zwracanych przez model.

In [ ]:
# wyświetlenie obrazu
input_image = Image.open(filename)
imgplot = plt.imshow(input_image)
plt.show()

input_tensor = preprocess(input_image)
# dodanie dodatkowego wymiaru odpowiadającego kolejnym obrazom
input_batch = input_tensor.unsqueeze(0)

#if torch.cuda.is_available():
#    input_batch = input_batch.to('cuda')
#    model.to('cuda')

with torch.no_grad():
    output = model(input_batch)

probabilities = #TODO ....

#wypisz najbardziej pewne kategorie dla obrazu
top5_prob, top5_catid = torch.topk(probabilities, 5)
#usunięcie pierwszego wymiaru = 1
top5_prob = top5_prob[0]
top5_catid = top5_catid[0]

for i in range(top5_prob.size(0)):
    print(categories[top5_catid[i]], "category id = ", top5_catid[i], "probability ", top5_prob[i].item())

#2. Wizualizacja działania sieci CNN za pomocą metoty opartej na gradiencie - GradCam.

Skorzystamy w tym celu z biblioteki: https://github.com/utkuozbulak/pytorch-cnn-visualizations. Sama metoda została opisana w artykule: https://arxiv.org/pdf/1610.02391.pdf.

In [ ]:
# import biblioteki z której będziemy korzystać
!git clone https://github.com/utkuozbulak/pytorch-cnn-visualizations
os.listdir(".")
os.chdir("pytorch-cnn-visualizations/src")


In [ ]:
# import odpowiednich funkcji
from gradcam import GradCam

from misc_functions import apply_colormap_on_image
from misc_functions import preprocess_image, convert_to_grayscale, save_gradient_images

In [ ]:
# WIZUALIZACJA:
img_as_var = Variable(input_batch, requires_grad=True)
# wywołanie funkcji GradCam
# TODO- można modyfikować argument target_layer
grad_cam = GradCam(model, target_layer=23)

# generowanie maski
label = top5_catid[0] # gradient obliczymy względem klasy przewidzianej przez model
cam = grad_cam.generate_cam(img_as_var, label)

# obraz wejściowy (do wizualizacji)
input_img_cam = preprocess_img(input_image)

heatmap, heatmap_on_image = apply_colormap_on_image(input_img_cam, cam, 'jet')

print("wizualizacja dla klasy: ", categories[label])

imgplot = plt.imshow(heatmap)
plt.show()

heat = plt.imshow(heatmap_on_image)
plt.show()

# Zadanie 2 [4 pkt.]
Poeskperymentuj z tą metodą wizualizacji działania sieci.
Przeanalizuj działanie algorytmu dla powyższego obrazu dla 3 najbardziej prawdopodobnych klas. Podobną analizę wykonaj dla trzech kolejnych dowolnych obrazów. Postaraj się przetestować model również dla sytuacji gdy mamy 2 obiekty na obrazie. Zwróć uwagę, że niektóre klasy nie występują w zbiorze ImageNet (np. człowiek).
Jak parametr target_layer wpływa na wynik wizualizacji?

W sprawozdaniu zapisz swoje obserwcje.

#3. Wizualizacja przez okluzje

W tym ćwiczeniu zajmiejmy się metodą wizualizacji przez okluzje. W pętli, pewne fragmenty obrazu będą kolejno przesłaniane przez szary prostokąt i zapisywana będzie pewnośc modelu, dla tak zmodyfikowanego obrazu. Dzięki temu można się dowiedzieć, które fragmenty obrazu pozytywnie lub negatywnie wpływają na decyzje modelu.

In [ ]:
# Funkcja dostępna w repozytorium: https://github.com/Niranjankumar-c/DeepLearning-PadhAI
iska. Wykaz
def occlusion(model, image, label, occ_size = 50, occ_stride = 50, occ_pixel = 0.5):

    width, height = image.shape[-2], image.shape[-1]

    # rozmiar mapy wyjściowej
    output_height = int(np.ceil((height-occ_size)/occ_stride))
    output_width = int(np.ceil((width-occ_size)/occ_stride))

    heatmap = torch.zeros((output_height, output_width))

    # iteracja po kolejnych okluzjach
    idx = 0
    for h in range(0, height):
        for w in range(0, width):

            h_start = h*occ_stride
            w_start = w*occ_stride
            h_end = min(height, h_start + occ_size)
            w_end = min(width, w_start + occ_size)

            if (w_end) >= width or (h_end) >= height:
                continue

            if idx % 25 == 0:
                print("iter {}".format(idx))

            input_image = image.clone().detach()

            #replacing all the pixel information in the image with occ_pixel(grey) in the specified location
            input_image[:, :, w_start:w_end, h_start:h_end] = occ_pixel

            #run inference on modified image
            output = model(input_image)
            output = nn.functional.softmax(output, dim=1)
            prob = output.tolist()[0][label]

            #setting the heatmap location to probability value
            heatmap[h, w] = prob
            idx += 1

    return heatmap

label = top5_catid[0]
heatmap = occlusion(model, img_as_var, label, 32, 14)

# wyświetlenie heatmapy
imgplot = sns.heatmap(heatmap, xticklabels=False, yticklabels=False, vmin=0.0, vmax=1.0)
figure = imgplot.get_figure()
figure.savefig('svm_conf.png', dpi=400)

#Zadanie 3 [2 pkt.]
Przetestuj wybraną metodę dla dwóch różnych obrazów i dwóch różnych klas. Zapisz obserwacje. Porównaj działanie z metodą GradCam.

#4. Przykłady mylące.
W tym ćwiczeniu zajmiemy się generowaniem przykładów mylących. Poprzez dodadnie niewielkiego, specjalnie przygotowanego szumu, znacząco zmienimy predykcję modelu.

Wykorzystana zostanie metoda Fast Gradient Sign Method (FGSM), której kod jest dostępny pod linkiem: https://pytorch.org/tutorials/beginner/fgsm_tutorial.html.

In [ ]:
# Metoda FGSM
def fgsm_attack(image, epsilon, data_grad):
    # pobranie znaku gradientu dla każdego piksela
    sign_data_grad = data_grad.sign()
    # Modyfikacja obrazu wejściowego zgodnie z gradientem o epsilon
    perturbed_image = image + epsilon*sign_data_grad
    return perturbed_image

In [ ]:
# ustawiamy możliwość propagacji gradientu do obrazu
img_as_var.requires_grad = True

# uruchomienie modelu
output = model(img_as_var)
init_pred = output.max(1, keepdim=True)[1] # indeks przewidywanej klasy

init_pred = init_pred[0]
# oblicznie funkcji kosztu
loss = F.nll_loss(output, init_pred)

# wyzerowanie gradientów
model.zero_grad()

# propagacja wsteczna
loss.backward()

# gradient względem obrazu
data_grad = img_as_var.grad.data

# wywołanie ataku z wartością epsilon
epsilon = .01
perturbed_data = fgsm_attack(img_as_var, epsilon, data_grad)

# klasyfikacja zmodyfikowanego obrazu
output = model(perturbed_data)

# nowa predykcja modelu
final_pred = output.max(1, keepdim=True)[1]

if init_pred != final_pred:
  print("Atak skuteczny! Poprzednio przewidywana klasa = {}, \
    przewidywana klasa dla zmodyfikowanego obrazu={}".format(init_pred, final_pred))
else:
  print("Atak nieskuteczny. Predykcja modelu się nie zmieniła. Zwiększ wartośc epsilon.")

perturbed_img = perturbed_data
inv_normalize = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.255],
    std=[1/0.229, 1/0.224, 1/0.255]
)
inv_tensor = inv_normalize(perturbed_img).detach().numpy()[0, :]

perturbed_img = inv_tensor.transpose(1,2,0)

#transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),

orig = plt.imshow(input_img_cam)
plt.show()

print(perturbed_img.shape)
heat = plt.imshow(perturbed_img)
plt.show()

# Zadanie 4 [3 pkt.]

Poeksperymentuj z przykładami mylącymi. Przygotuj przykłady mylące dla jednego / dwóch obrazów, a następnie wykorzystaj metody wizualizacji aby zrozumieć, które części obrazu są odpowiedzialne za nową (błędną) oraz starą (poprawną) predykcję.
Co Twoim zdaniem przykłady mylące (i to jak łatwo je wygenerować) mówią Ci na temat sieci CNN.